# Streaming related-gene selection

For every target dependency gene, this notebook:

1. matches CRISPR Gene Effect and RNA expression by `ModelID`;
2. tests every expression gene with Pearson, Spearman, and Chi-square;
3. applies Benjamini–Hochberg FDR separately to each method;
4. selects the union of Top-K significant genes;
5. immediately appends one compact record to a shared JSONL file;
6. resumes from that JSONL file after a restart and skips completed targets.

Each JSONL line is a complete result for one target gene. Statistical association does not establish causality.

In [1]:
import json
import os
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. Configuration

`PRIORITY_TARGET_GENES` are processed first. The remaining genes follow their original CRISPR column order. `MAX_NEW_TARGETS` can limit a test run; `None` processes every unfinished target.

In [2]:
EXPRESSION_PATH = Path("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")
GENE_EFFECT_PATH = Path("CRISPRGeneEffect.csv")
OUTPUT_DIR = Path("static_gene_selection")
JSONL_PATH = OUTPUT_DIR / "target_related_genes.jsonl"

PRIORITY_TARGET_GENES = [
    "ATPSCKMT (134145)",
]
TARGET_START_INDEX = 0
TARGET_END_INDEX = None
MAX_NEW_TARGETS = None

TOP_K = 20
FDR_ALPHA = 0.05
MIN_N = 20
N_ESSENTIALITY_QUANTILES = 6
EXCLUDE_TARGET_FROM_FEATURES = True

## 2. Helpers

In [3]:
def gene_symbol(column_name):
    return re.sub(r"\s+\(\d+\)$", "", str(column_name)).strip()


def build_gene_map(columns):
    mapping = {}
    for column in columns:
        symbol = gene_symbol(column)
        if symbol not in mapping:
            mapping[symbol] = column
    return mapping


def resolve_gene_column(df, gene):
    if gene in df.columns:
        return gene
    return build_gene_map(df.columns)[gene_symbol(gene)]


def load_gene_relations(path):
    relations = {}
    if not path.exists():
        return relations

    with path.open(encoding="utf-8") as file:
        for line in file:
            relations.update(json.loads(line))
    return relations


def append_gene_relation(path, target_gene, related_genes):
    record = {target_gene: related_genes}
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")
        file.flush()
        os.fsync(file.fileno())

## 3. Load and align data

In [4]:
expression = pd.read_csv(EXPRESSION_PATH, low_memory=False)
gene_effect = pd.read_csv(GENE_EFFECT_PATH, low_memory=False)
gene_effect = gene_effect.rename(columns={"Unnamed: 0": "ModelID"})

default_mask = (
    expression["IsDefaultEntryForModel"]
    .astype(str)
    .str.lower()
    .isin(["yes", "true", "1"])
)
expression = expression.loc[default_mask].drop_duplicates("ModelID")

expression_gene_columns = [
    column
    for column in expression.columns
    if re.search(r"\(\d+\)$", str(column))
]

common_models = sorted(
    set(expression["ModelID"].dropna())
    & set(gene_effect["ModelID"].dropna())
)

expression_matrix = (
    expression.loc[
        expression["ModelID"].isin(common_models),
        ["ModelID", *expression_gene_columns],
    ]
    .set_index("ModelID")
    .sort_index()
)

gene_effect_matrix = (
    gene_effect[gene_effect["ModelID"].isin(common_models)]
    .drop_duplicates("ModelID")
    .set_index("ModelID")
    .sort_index()
)

all_target_genes = gene_effect_matrix.columns.tolist()
selected_target_genes = all_target_genes[TARGET_START_INDEX:TARGET_END_INDEX]
TARGET_GENES = list(dict.fromkeys([
    *PRIORITY_TARGET_GENES,
    *selected_target_genes,
]))

print("Expression matrix:", expression_matrix.shape)
print("Gene Effect matrix:", gene_effect_matrix.shape)
print("Targets in this run:", len(TARGET_GENES))
print("First targets:", TARGET_GENES[:5])

Expression matrix: (1140, 19215)
Gene Effect matrix: (1140, 18531)
Targets in this run: 18531
First targets: ['ATPSCKMT (134145)', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)']


## 4. Statistical tests

`chi2` and `chi2_p` are `NaN` when a valid contingency table cannot be built. `chi2_status` records the reason.

In [5]:
def safe_pearson(x, y, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)
    if len(xx) < min_n or xx.nunique() < 2 or yy.nunique() < 2:
        return np.nan, np.nan, len(xx)
    coefficient, p_value = stats.pearsonr(xx, yy)
    return coefficient, p_value, len(xx)


def safe_spearman(x, y, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)
    if len(xx) < min_n or xx.nunique() < 2 or yy.nunique() < 2:
        return np.nan, np.nan, len(xx)
    coefficient, p_value = stats.spearmanr(xx, yy)
    return coefficient, p_value, len(xx)


def safe_chi_square(x, y, n_quantiles=N_ESSENTIALITY_QUANTILES, min_n=MIN_N):
    mask = x.notna() & y.notna()
    xx = x[mask].astype(float)
    yy = y[mask].astype(float)

    if len(xx) < min_n:
        return np.nan, np.nan, len(xx), np.nan, "insufficient_n"
    if xx.nunique() < 2:
        return np.nan, np.nan, len(xx), np.nan, "constant_expression"
    if yy.nunique() < 2:
        return np.nan, np.nan, len(xx), np.nan, "constant_gene_effect"

    median = xx.median()
    expression_bin = pd.Series(
        np.where(xx <= median, "low", "high"),
        index=xx.index,
    )

    try:
        effect_bin = pd.qcut(yy, q=n_quantiles, duplicates="drop")
    except ValueError:
        return np.nan, np.nan, len(xx), median, "gene_effect_qcut_failed"

    if effect_bin.nunique() < 2:
        return np.nan, np.nan, len(xx), median, "single_gene_effect_bin"

    table = pd.crosstab(expression_bin, effect_bin)
    if table.shape[0] < 2:
        return np.nan, np.nan, len(xx), median, "single_expression_bin"
    if table.shape[1] < 2:
        return np.nan, np.nan, len(xx), median, "single_gene_effect_bin"

    statistic, p_value, _, _ = stats.chi2_contingency(table)
    return statistic, p_value, len(xx), median, "ok"


def add_bh_fdr(df, p_column, fdr_column):
    result = df.copy()
    result[fdr_column] = np.nan
    mask = result[p_column].notna()
    result.loc[mask, fdr_column] = multipletests(
        result.loc[mask, p_column].values,
        alpha=FDR_ALPHA,
        method="fdr_bh",
    )[1]
    return result

## 5. Analyze one target gene

In [6]:
def analyze_target_gene(target_gene):
    target_column = resolve_gene_column(gene_effect_matrix, target_gene)
    target_symbol = gene_symbol(target_column)
    y = gene_effect_matrix[target_column].rename("GeneEffect")
    valid_models = y[y.notna()].index.intersection(expression_matrix.index)
    y = y.loc[valid_models]
    target_expression = expression_matrix.loc[valid_models]

    feature_columns = list(target_expression.columns)
    if EXCLUDE_TARGET_FROM_FEATURES:
        feature_columns = [
            column
            for column in feature_columns
            if gene_symbol(column) != target_symbol
        ]

    rows = []
    for column in feature_columns:
        x = pd.to_numeric(target_expression[column], errors="coerce")
        pearson_r, pearson_p, pearson_n = safe_pearson(x, y)
        spearman_rho, spearman_p, spearman_n = safe_spearman(x, y)
        chi2, chi2_p, chi2_n, expression_median, chi2_status = safe_chi_square(x, y)
        rows.append({
            "gene": gene_symbol(column),
            "expression_column": column,
            "n": min(pearson_n, spearman_n, chi2_n),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "chi2": chi2,
            "chi2_p": chi2_p,
            "chi2_status": chi2_status,
            "expression_median": expression_median,
        })

    results = pd.DataFrame(rows)
    results = add_bh_fdr(results, "pearson_p", "pearson_fdr")
    results = add_bh_fdr(results, "spearman_p", "spearman_fdr")
    results = add_bh_fdr(results, "chi2_p", "chi2_fdr")

    results["pearson_significant"] = results["pearson_fdr"] < FDR_ALPHA
    results["spearman_significant"] = results["spearman_fdr"] < FDR_ALPHA
    results["chi2_significant"] = results["chi2_fdr"] < FDR_ALPHA

    pearson_top = (
        results[results["pearson_significant"]]
        .assign(abs_score=lambda data: data["pearson_r"].abs())
        .sort_values(["pearson_fdr", "abs_score"], ascending=[True, False])
        .head(TOP_K)
    )
    spearman_top = (
        results[results["spearman_significant"]]
        .assign(abs_score=lambda data: data["spearman_rho"].abs())
        .sort_values(["spearman_fdr", "abs_score"], ascending=[True, False])
        .head(TOP_K)
    )
    chi2_top = (
        results[results["chi2_significant"]]
        .sort_values(["chi2_fdr", "chi2"], ascending=[True, False])
        .head(TOP_K)
    )

    pearson_genes = set(pearson_top["gene"])
    spearman_genes = set(spearman_top["gene"])
    chi2_genes = set(chi2_top["gene"])
    selected_genes = pearson_genes | spearman_genes | chi2_genes
    selected = results[results["gene"].isin(selected_genes)].copy()
    selected["selected_by_pearson"] = selected["gene"].isin(pearson_genes)
    selected["selected_by_spearman"] = selected["gene"].isin(spearman_genes)
    selected["selected_by_chi2"] = selected["gene"].isin(chi2_genes)
    selected["n_methods"] = selected[
        ["selected_by_pearson", "selected_by_spearman", "selected_by_chi2"]
    ].sum(axis=1)
    selected["best_fdr"] = selected[
        ["pearson_fdr", "spearman_fdr", "chi2_fdr"]
    ].min(axis=1)
    selected = selected.sort_values(
        ["n_methods", "best_fdr"],
        ascending=[False, True],
    ).reset_index(drop=True)

    report = {
        "target_gene": target_column,
        "target_symbol": target_symbol,
        "n_models": len(y),
        "parameters": {
            "top_k": TOP_K,
            "fdr_alpha": FDR_ALPHA,
            "min_n": MIN_N,
            "essentiality_quantiles": N_ESSENTIALITY_QUANTILES,
            "target_excluded_from_features": EXCLUDE_TARGET_FROM_FEATURES,
        },
        "summary": {
            "tested_expression_genes": len(results),
            "pearson_significant": int(results["pearson_significant"].sum()),
            "spearman_significant": int(results["spearman_significant"].sum()),
            "chi2_significant": int(results["chi2_significant"].sum()),
            "selected_related_genes": len(selected),
        },
        "related_genes": selected["expression_column"].tolist(),
    }
    return report, results, selected

## 6. Process, save, and resume

The result is written immediately after each target. If execution stops, rerun this cell or the whole notebook: targets already present in JSONL are skipped.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_relations = load_gene_relations(JSONL_PATH)
pending_targets = [
    gene
    for gene in TARGET_GENES
    if resolve_gene_column(gene_effect_matrix, gene) not in processed_relations
]

if MAX_NEW_TARGETS is not None:
    pending_targets = pending_targets[:MAX_NEW_TARGETS]

print("Already processed:", len(processed_relations))
print("Pending in this run:", len(pending_targets))
print("JSONL:", JSONL_PATH.resolve())

for position, target_gene in enumerate(pending_targets, start=1):
    target_column = resolve_gene_column(gene_effect_matrix, target_gene)
    print(f"[{position}/{len(pending_targets)}] Analyzing {target_column}...")
    report, _, _ = analyze_target_gene(target_column)
    append_gene_relation(
        JSONL_PATH,
        report["target_gene"],
        report["related_genes"],
    )
    processed_relations[report["target_gene"]] = report["related_genes"]
    print(
        f"Saved {report['target_gene']}: "
        f"{len(report['related_genes'])} related genes"
    )

Already processed: 0
Pending in this run: 1
JSONL: /Users/nident/Desktop/JOB/Cancer Cell Line Encyclopedia (CCLE)/data/static_gene_selection/target_related_genes.jsonl
[1/1] Analyzing ATPSCKMT (134145)...


Saved ATPSCKMT (134145): 33 related genes


## 7. Current progress

The JSONL file is the primary output consumed by the pipeline. One line represents one fully processed target gene.

In [8]:
processed_relations = load_gene_relations(JSONL_PATH)
print("Processed target genes:", len(processed_relations))
print("Total target genes:", len(all_target_genes))
print("Remaining:", len(set(all_target_genes) - set(processed_relations)))
print("JSONL:", JSONL_PATH.resolve())

Processed target genes: 1
Total target genes: 18531
Remaining: 18530
JSONL: /Users/nident/Desktop/JOB/Cancer Cell Line Encyclopedia (CCLE)/data/static_gene_selection/target_related_genes.jsonl


## 8. Inspect saved results

In [9]:
for target_gene in list(processed_relations)[-3:]:
    print(json.dumps(
        {target_gene: processed_relations[target_gene]},
        ensure_ascii=False,
        indent=2,
    ))

{
  "ATPSCKMT (134145)": [
    "TEDC1 (283643)",
    "UTRN (7402)",
    "H2AC20 (8338)",
    "ATG101 (60673)",
    "RPS18 (6222)",
    "MALSU1 (115416)",
    "CENPN (55839)",
    "EXOSC4 (54512)",
    "GLTPD2 (388323)",
    "ONECUT1 (3175)",
    "KLHL25 (64410)",
    "CEACAM20 (125931)",
    "CENPI (2491)",
    "RPS11 (6205)",
    "FOLH1 (2346)",
    "CDC6 (990)",
    "CMC2 (56942)",
    "RPL24 (6152)",
    "MCM8 (84515)",
    "BTG1 (694)",
    "CIROP (100128908)",
    "RHBDD3 (25807)",
    "KIF23 (9493)",
    "IMPA2 (3613)",
    "FAM177A1 (283635)",
    "GPR157 (80045)",
    "RAD54B (25788)",
    "FNTB (2342)",
    "TYRO3 (7301)",
    "DSCC1 (79075)",
    "ENPP1 (5167)",
    "UCK1 (83549)",
    "ITPK1 (3705)"
  ]
}
